A correct **data pipeline** must be:
1. **Deterministic** – same input -> same output
1. **Ordered** – steps must happen in the right sequence
1. **Reusable** – train & inference share logic
1. **Auditable** – easy to debug
1. **Fail-fast** – breaks loudly when assumptions fail

### The Canonical Pipeline Order
```pgsql
Load
 → Schema Validation
 → Cleaning
 → EDA checks (lightweight)
 → Feature Engineering
 → Post-feature Validation
 → Output
```

### Separate Responsibilities
#### Everything inline (Bad)
```python
df["age"] = df["age"].clip(0, 120)
df["salary_log"] = np.log(df["salary"])
df = df.dropna()
```

Can't infer:
- what was invalid
- what was imputed
- what was dropped
- why

#### Explicit stages (good)
```python
df = validate_schema(df)
df = clean_data(df)
df = engineer_features(df)
validate_features(df)
```

### Schema Contracts (Pipeline Backbone)
```python
EXPECTED_SCHEMA = {
    "user_id": "int",
    "age": "float",
    "salary": "float",
    "city": "category",
    "signup_date": "datetime64[ns]"
}
```

**Validation:**
```python
def validate_schema(df):
    assert set(df.columns) == set(EXPECTED_SCHEMA)
    return df
```

### Cleaning as Pure Functions
#### Bad (side effects)
```python
def clean_age(df):
    df["age"].fillna(df["age"].median(), inplace=True)
```

#### Good (pure, testable)
```python
def clean_age(df):
    df = df.copy()
    invalid = ~df["age"].between(0, 120)
    df.loc[invalid, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())
    return df
```

### Feature Engineering Must Be Deterministic
#### Dangerous
```python
df["salary_norm"] = df["salary"] / df["salary"].max()
```
Why bad?
   - max changes over time
   - inference ≠ training

#### Safe pattern
```python
SALARY_MAX = df["salary"].max()

df["salary_norm"] = df["salary"] / SALARY_MAX
```

### Train vs Inference Split
#### Wrong
```python
median_age = df["age"].median()
```
This changes when new data arrives.

#### Correct
```python
# training
AGE_MEDIAN = train_df["age"].median()

# inference
df["age"] = df["age"].fillna(AGE_MEDIAN)
```
Training learns parameters. Inference applies them.

### Pipeline Configuration
```python
CONFIG = {
    "valid_cities": {"Pune", "Mumbai", "Delhi"},
    "max_age": 120,
    "salary_floor": 0
}
```
Pipelines should change via **config, not code edits.**

### Lightweight Pipelien Skeleton
```python
def run_pipeline(df, config):
    df = validate_schema(df)
    df = clean_data(df, config)
    df = engineer_features(df, config)
    validate_features(df, config)
    return df
```
Industry-grade structure.

### Logging & Metrics 
Track:
- rows in / out
- % missing before & after
- rows dropped
- invalid values fixed

```python
print("Rows before:", len(df))
```

### Reproducibility Rules (MEMORIZE)
1. No random operations without seed
1. No recomputation of learned statistics
1. Same code path for train & inference
1. Immutable configs
1. Deterministic ordering

### Base Dataset

In [30]:
import pandas as pd
import numpy as np

np.random.seed(42)

raw_df = pd.DataFrame({
    "user_id": [1, 2, 2, 4, 5, None, 6],
    "age": [25, -3, 30, 200, None, 40, 28],
    "salary": [50000, None, -1000, 70000, 65000, None, 90000],
    "city": [" pune ", "Mumbai", "DELHI", "Puna", None, "Mumbai", "Pune"],
    "signup_date": [
        "2024-01-01",
        "2024-01-05",
        "invalid_date",
        "2026-03-01",
        None,
        "2024-01-10",
        "2024-01-03"
    ]
})

In [31]:
CONFIG = {
    "valid_cities": {"Pune", "Mumbai", "Delhi"},
    "max_age": 120,
    "min_salary": 0
}

### Exercise 1
Write a function:
```python
def validate_schema(df):
    ...
```
Rules:
   - Required columns:
```python
{"user_id", "age", "salary", "city", "signup_date"}
```
   - No extra columns allowed
   - Raise assertion errors if schema is violated

In [32]:
def validate_schema(df):
    required_cols = {"user_id", "age", "salary", "city", "signup_date"}
    assert set(df.columns) == required_cols, "Schema mismatch"
    return df

### Exercise 2
Write:
```python
def handle_primary_key(df):
    ...
```
Rules:
- Drop rows with missing `user_id`
- Do NOT deduplicate yet
- Log number of rows removed      

Explain why deduplication is delayed.

In [33]:
def handle_primary_key(df):
    before = len(df)
    df = df.dropna(subset=["user_id"])
    after = len(df)
    print(f"Dropped {before - after} rows with missing user_id")
    return df

Because deduplication depends on signup_date, which isn’t cleaned yet.

### Exercise 3
Implement **separate functions:**
1. Clean `age`
```python
def clean_age(df, config):
    ...
```
Rules:
   - Valid range: `0 -> config["max_age"]`
   - Invalid values -> `NaN`
   - Impute missing with **median (computed inside function)**
2. Clean `salary`
```python
def clean_salary(df, config):
    ...
```
Rules:
   - Salary `< config["min_salary"]` -> `NaN`
   - Missing salary:
       - if city is known -> fill with **city median**
       - else → keep `NaN`
3. Clean `city`
```python
def clean_city(df, config):
    ...
```
Rules:
   - Strip whitespace
   - Normalize case (Title Case)
   - Enforce domain using `config["valid_cities"]`
   - All others -> `"Unknown"`
4. Clean `signup_date`
```python
def clean_signup_date(df):
    ...
```
Rules:
   - Parse safely (`errors="coerce"`)
   - Drop rows with:
       - missing dates
       - future dates

Explain why dropping is safer here.

In [34]:
def clean_age(df, config):
    df = df.copy()
    invalid = ~df["age"].between(0, config["max_age"])
    df.loc[invalid, "age"] = np.nan
    df["age"] = df["age"].fillna(df["age"].median())
    return df

In [35]:
def clean_salary(df, config):
    df = df.copy()
    df.loc[df["salary"] < config["min_salary"], "salary"] = np.nan
    df["salary"] = df.groupby("city")["salary"].transform(
        lambda x: x.fillna(x.median())
        )
    return df

In [36]:
def clean_city(df, config):
    df = df.copy()
    df["city"] = (
        df["city"]
        .str.strip()
        .str.title()
    )
    df.loc[~df["city"].isin(config["valid_cities"]), "city"] = "Unknown"
    return df

In [37]:
def clean_signup_date(df):
    df = df.copy()
    df["signup_date"] = pd.to_datetime(df["signup_date"], errors="coerce")

    df = df.dropna(subset=["signup_date"])
    df = df[df["signup_date"] <= pd.Timestamp.today()]
    return df

Why drop instead of fix?
- Dates define **time truth**
- Shifting creates **temporal lies**
- Safer to fail than hallucinate

### Exercise 4
Write:
```python
def deduplicate_users(df):
    ...
```
Rules:
- One row per `user_id`
- Keep row with **latest signup_date**

In [38]:
def deduplicate_users(df):
    df = (
        df.sort_values("signup_date")
        .drop_duplicates("user_id", keep="last")
    )
    return df

### Exercise 5
Write:
```python
def engineer_features(df):
    ...
```
Create:
- `salary_log` -> `log(salary)`
- `is_high_earner` -> salary > 80000

Rules:
- No random operations
- No recomputation of statistics

In [39]:
def engineer_features(df):
    df = df.copy()
    df["salary_log"] = np.log(df["salary"])
    df["is_high_earner"] = df["salary"] > 80000
    return df

### Exercise 6
Write:
```python
def validate_features(df, config):
    ...
```
Assert:
- `user_id` unique & non-null
- `age` within valid range
- `salary_log` has no `inf`
- `city` contains only allowed + `"Unknown"`
- No future dates

In [40]:
def validate_features(df, config):
    assert df["user_id"].notna().all()
    assert df["user_id"].is_unique
    assert df["age"].between(0, config["max_age"]).all()
    assert (df["salary"].ge(0) | df["salary"].isna()).all()
    assert np.isfinite(df["salary_log"].dropna()).all()
    assert set(df["city"]).issubset(config["valid_cities"] | {"Unknown"})
    assert df["signup_date"].max() <= pd.Timestamp.today()

### Exercise 7
Write the final pipeline:
```python
def run_pipeline(df, config):
    ...
```
Order must be:
```
validate_schema
→ handle_primary_key
→ clean_city
→ clean_age
→ clean_salary
→ clean_signup_date
→ deduplicate_users
→ engineer_features
→ validate_features
```

In [41]:
def run_pipeline(df, config):
    df = validate_schema(df)
    df = handle_primary_key(df)
    df = clean_city(df, config)
    df = clean_age(df, config)
    df = clean_salary(df, config)
    df = clean_signup_date(df)
    df = deduplicate_users(df)
    df = engineer_features(df)
    validate_features(df, config)
    return df

| Step                    | Reason                            |
| ----------------------- | --------------------------------- |
| Schema first            | Cleaning assumes structure        |
| City before salary      | Salary imputation depends on city |
| Dates before dedup      | Dedup uses signup_date            |
| Features after cleaning | Avoid propagating errors          |
| Validation last         | Catch hidden bugs                 |


In [42]:
final_df = run_pipeline(raw_df, CONFIG)
final_df

Dropped 1 rows with missing user_id


/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Users/jatinrokde/CodeBase/SynapseWorks/MLForge/.venv/lib/python3.13/site-packages/numpy/lib/_nanfunctions_impl.py:1214: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)


,user_id,age,salary,city,signup_date,salary_log,is_high_earner
0,1.0,25.0,50000.0,Pune,2024-01-01,10.819778,False
6,6.0,28.0,90000.0,Pune,2024-01-03,11.407565,True
1,2.0,28.0,NaN,Mumbai,2024-01-05,NaN,False
